# 03: Data Science & Deep Learning Project Issues & Solutions (A-Z Master Guide)
### (දත්ත විද්‍යා සහ Deep Learning ව්‍යාපෘතිවලදී මතුවන ගැටලු, සබෑ සිදුවීම් සහ ප්‍රායෝගික විසඳුම්)

Deep Learning ව්‍යාපෘතියක් සාර්ථකව නිමකිරීම කියන්නේ නිකම්ම Code පේළි ටිකක් ලියන එක නෙවෙයි. Real-world Data Science projects වලදී Data Preprocessing, Training, GPU Hardware Optimization, සහ Production Deployment කියන සෑම අදියරකයදීම අපි නොසිතන ආකාරයේ සංකීර්ණ Bugs සහ Failures මතු වෙනවා.

මෙම Notebook එකෙන් අපි කතා කරන්නේ ඒ වගේ **සැබෑ ප්‍රායෝගික සිදුවීම් (Real Production Incidents)**, ඒවා මතු වූ ආකාරය, ඊට හේතු (Root Cause), සහ Python/TensorFlow මගින් ඒවා නිරාකරණය කරගන්නා ආකාරය පිළිබඳ විස්තරාත්මක සිංහල මාර්ගෝපදේශයයි.


--- 
## 1. Data Level Issues & Real-world Incidents (දත්ත මට්ටමේ ගැටලු)

### 1.1 Incident Story: The 99% Accuracy Trap (Severe Class Imbalance Incident)

**සැබෑ සිදුවීම (Real Incident):**
මූල්‍ය ආයතනයක (Financial Firm) Data Science කණ්ඩායමක් විසින් වංචනික SMS (Fraudulent SMS) හඳුනාගැනීම සඳහා Model එකක් නිර්මාණය කළා. පුහුණුවෙන් පසු Model එක 98.5%ක අතිවිශාල Accuracy එකක් පෙන්නුම් කළා! නමුත් මෙය Production එකට දැමූ පසු, පාරිභෝගිකයින් දිනපතාම ලැබෙන Fraud SMS නිසා අසරණ වූ අතර Model එක එකදු Fraud SMS එකක්වත් හඳුනාගැනීමට සමත් වුණේ නැහැ.

**සිදුවීමට හේතුව (Root Cause Analysis):**
Dataset එක පරීක්ෂා කිරීමේදී හෙළිවුණේ එහි පැවති SMS 100,000කින් 98,500ක්ම සාමාන්‍ය පණිවිඩ (Ham) වූ අතර 1,500ක් පමණක් Fraud SMS වූ බවයි. Model එක ඉතා කූට ලෙස කිසිදු මහන්සියක් නොවී සෑම පණිවිඩයකටම "Ham" (0) කියා Predict කිරීමට ඉගෙනගෙන තිබුණා! එවිට එයට ඉබේම 98.5%ක Accuracy එකක් ලැබෙනවා. මෙය Deep Learning හි **Class Imbalance Trap** එක ලෙස හඳුන්වනවා.

**විසඳුම (Solution Strategy):**
1. Model fit කිරීමේදී `compute_class_weight` මගින් සුළුතර පංතියට (Fraud SMS) වැඩි බරක් (Higher Penalty Weight) ලබාදීම.
2. Accuracy වෙනුවට **Precision, Recall, සහ F1-Score** මැනීම.


In [1]:
import numpy as np
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight

# 1. අසමතුලිත දත්ත (9850 normal vs 150 fraud)
y_train_imbalanced = np.array([0]*9850 + [1]*150)

# 2. Class Weights ගණනය කිරීම
weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_imbalanced),
    y=y_train_imbalanced
)
class_weight_dict = dict(enumerate(weights))
print("Calculated Class Weights (Notice how minority class gets much higher weight):", class_weight_dict)

# 3. Model Training හිදී class_weight ලබාදීම:
# model.fit(X_train, y_train, class_weight=class_weight_dict, epochs=10)


Calculated Class Weights (Notice how minority class gets much higher weight): {0: np.float64(0.5076142131979695), 1: np.float64(33.333333333333336)}


--- 
### 1.2 Incident Story: The Silent Test Leakage Disaster (Data Leakage Incident)

**සැබෑ සිදුවීම (Real Incident):**
ගෙවල් මිල පුරෝකථනය කරන (Housing Price Prediction) ව්‍යාපෘතියකදී Data Scientist කෙනෙක් සකස් කළ Model එක Testing වලදී RMSE 0.05 වැනි ඉතා කුඩා අගයක් පෙන්නුම් කළා. නමුත් මෙය Client ට ලබාදී Live Data මත ක්‍රියාත්මක කරන විට RMSE එක 0.95 දක්වා අධික ලෙස ඉහළ ගොස් ආකෘතිය සම්පූර්ණයෙන්ම අසාර්ථක වුණා.

**සිදුවීමට හේතුව (Root Cause Analysis):**
පරීක්ෂා කිරීමේදී හෙළිවුණේ Data Preprocessing පියවරේදී අත් වැරැද්දකින් `StandardScaler().fit_transform(X_all)` ලෙස **සමස්ත Dataset එකටම Split කිරීමට ප්‍රථමයෙන් Scaling සිදුකර ඇති බවයි**. එමගින් Test set එකෙහි Mean සහ Standard Deviation අගයන් Train set එක තුළට කාන්දු වී (Data Leakage) තිබුණා. එම නිසා Test set එකෙහි උත්තර Model එක කලින්ම දැනගෙන සිටියා.

**විසඳුම (Solution Strategy):**
Scaling සහ Vectorization සෑම විටම **`X_train` සඳහා පමණක් fit කර**, එම fitted parameters වලින් `X_test` transform කිරීම (නැතහොත් Keras `Normalization` In-Model layer එක යෙදීම).


In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Sample Data
X_dummy = np.random.randn(1000, 5) * 50 + 100
y_dummy = np.random.randn(1000)

# 1. පළමුව Train / Test ලෙස වෙන් කරන්න
X_train, X_test, y_train, y_test = train_test_split(X_dummy, y_dummy, test_size=0.2, random_state=42)

# 2. Scaler එක FIT කරන්නේ X_train සඳහා පමණි!
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train) # Fit and Transform
X_test_scaled = scaler.transform(X_test)       # ONLY Transform (No fit! Prevents Leakage)

print("Correct Preprocessing Implemented! Leakage Prevented.")


Correct Preprocessing Implemented! Leakage Prevented.


--- 
## 2. Model Training & Convergence Incidents (පුහුණු කිරීමේදී මතුවන ගැටලු)

### 2.1 Incident Story: The Midnight Loss NaN Collapse (Exploding Gradient & Numerical Instability)

**සැබෑ සිදුවීම (Real Incident):**
පැය 8ක් තිස්සේ Train වෙමින් තිබූ Deep Neural Network එකක් 5 වන Epoch එකට පැමිණි විගස එකපාරටම Loss එක `NaN` (Not a Number) බවට පත්වී, Accuracy එක 10% (Random guess) දක්වා බිඳ වැටුණා. Weights සියල්ලම `NaN` බවට පත්ව තිබූ බැවින් මුළු Training ක්‍රියාවලියම අපතේ ගියා.

**සිදුවීමට හේතුව (Root Cause Analysis):**
1. Learning Rate එක තරමක් වැඩිවීම සහ Unnormalized features පැවතීම නිසා Backpropagation වලදී Gradients අධික ලෙස වැඩිවී (Exploding Gradients) Float limit එක ඉක්මවා යාම.
2. Softmax cross-entropy loss එක ගණනය කිරීමේදී $log(0)$ සෑදීම නිසා Numerical Instability ඇතිවීම.

**විසඳුම (Solution Strategy):**
1. Optimizer එකට **`clipnorm=1.0`** එකතු කර Gradients පාලනය කිරීම.
2. Loss function එක සඳහා **`from_logits=True`** යෙදීම.
3. Activation සඳහා `LeakyReLU` සහ Proper Weight Initialization (`he_normal`) යෙදීම.


In [3]:
# Exploding Gradients සහ NaN Loss වළක්වන Safe Optimizer & Loss Setup
safe_adam = tf.keras.optimizers.Adam(
    learning_rate=0.001,
    clipnorm=1.0  # Gradient 1.0ට වඩා වැඩිවීම වළක්වයි
)

# Numerically stable loss function using logits
safe_loss = tf.keras.losses.CategoricalCrossentropy(from_logits=True)

print("Configured Gradient Clipping and Numerically Stable Logit Loss!")


Configured Gradient Clipping and Numerically Stable Logit Loss!


--- 
## 3. Hardware, Memory & Deployment Incidents (පද්ධති සහ Export කිරීමේ ගැටලු)

### 3.1 Incident Story: The GPU Resource Exhausted OOM Crash (GPU Memory Incident)

**සැබෑ සිදුවීම (Real Incident):**
High-resolution Computer Vision model එකක් GPU එකක් මත Train කිරීමට සූදානම් වූ විට, පළමු batch එකේදීම `ResourceExhaustedError: OOM when allocating tensor...` කියා මුළු program එකම crash වුණා.

**සිදුවීමට හේතුව (Root Cause Analysis):**
Batch size එක 128 ලෙස සකසා තිබූ අතර, High-resolution images සහ Convolutional activations මගින් GPU VRAM (Memory) එක එකපාරටම පිරී ඉතිරී යාමයි.

**විසඳුම (Solution Strategy):**
1. Batch Size එක 32 හෝ 16 දක්වා අඩුකිරීම.
2. **Mixed Precision Training (`mixed_float16`)** සක්‍රිය කිරීම (එමගින් memory භාවිතය 50%කින් අඩුවේ).
3. TensorFlow GPU Memory Growth Enable කිරීම.


In [4]:
# GPU Memory Growth & Mixed Precision Configuration
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU Dynamic Memory Growth Enabled!")
    except RuntimeError as e:
        print(e)

# Mixed Precision Policy (Float16 computation for 2x speed and half memory)
# tf.keras.mixed_precision.set_global_policy('mixed_float16')
print("Mixed Precision policy reduces GPU VRAM consumption by ~50%!")


Mixed Precision policy reduces GPU VRAM consumption by ~50%!


--- 
### 3.2 Incident Story: The Unknown Custom Layer Production Crash (Model Export Incident)

**සැබෑ සිදුවීම (Real Incident):**
Custom Residual Layer එකක් භාවිත කර සෑදූ Model එකක් train කර `.h5` file එකක් ලෙස save කර Production Server එකට යැවූ පසු, Server එකෙහි `tf.keras.models.load_model()` කරන විට `ValueError: Unknown layer: CustomResidualBlock` කියා Error එකක් ලබාදෙමින් Deployment එක බිඳ වැටුණා.

**සිදුවීමට හේතුව (Root Cause Analysis):**
Custom Layer එකෙහි `get_config()` method එක සකස් කර නොතිබීම සහ Keras Object Registry එකෙහි එම custom layer එක ලියාපදිංචි කර නොතිබීමයි.

**විසඳුම (Solution Strategy):**
Custom components සඳහා **`@tf.keras.utils.register_keras_serializable()`** යෙදීම සහ Keras 3 Native **`.keras`** format එක භාවිත කිරීම.


In [5]:
# Safe Registered Custom Layer
@tf.keras.utils.register_keras_serializable(package="MyCustomLayers")
class SafeCustomDense(tf.keras.layers.Layer):
    def __init__(self, units=32, **kwargs):
        super(SafeCustomDense, self).__init__(**kwargs)
        self.units = units
        self.dense = tf.keras.layers.Dense(units, activation='relu')
        
    def call(self, inputs):
        return self.dense(inputs)
        
    def get_config(self):
        config = super(SafeCustomDense, self).get_config()
        config.update({"units": self.units})
        return config

print("Custom Layer registered! Can now be saved and loaded anywhere seamlessly as .keras file.")


Custom Layer registered! Can now be saved and loaded anywhere seamlessly as .keras file.
